# 5.4 Duck Typing, Protocols & Composition

**Prerequisites:** 5.1 Python OOPs, 5.2 Payroll System, 5.3 Dataclasses & Enums  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Duck typing — the principle Python's whole object model rests on
- Why `isinstance` checks are usually the wrong instinct
- **ABCs vs Protocols** — nominal vs structural typing
- `typing.Protocol` and `@runtime_checkable`
- Dependency injection without a framework
- Choosing between an ABC, a Protocol, and nothing at all

---

## 1. Duck typing

> *"If it walks like a duck and quacks like a duck, it's a duck."*

Python does not ask **what an object is**. It asks **what an object can do**, at the moment
it tries to do it.

When you write `total = a + b`, Python does not check that `a` is a number. It looks for
`a.__add__` and calls it. When you write `for x in thing`, it does not check that `thing` is
a list — it looks for `thing.__iter__`.

This is why the `DisgruntledEmployee` in **5.2** worked with `PayrollSystem` despite sharing
no base class: it had a `calculate_payroll()` method, and that was the entire requirement.

### Why this matters in practice

Duck typing is what makes Python code composable. You can pass a `StringIO` anywhere a file
is expected, a generator anywhere a list is expected, a `Decimal` anywhere a number is
expected — **without those classes knowing about each other**.

**Real-world use case:** writing a function that takes "anything file-like" and having it
work with real files, in-memory buffers, network streams, and gzip wrappers, none of which
share a base class.

In [ ]:
import io


# ---- A function that never asks "what are you?" ----
def write_report(destination, rows: list[tuple[str, int]]) -> None:
    """Works with ANYTHING that has a .write() method."""
    destination.write("item,count\n")
    for name, count in rows:
        destination.write(f"{name},{count}\n")


data = [("errors", 12), ("warnings", 47), ("info", 903)]

# A real in-memory file
buffer = io.StringIO()
write_report(buffer, data)
print("StringIO:")
print(buffer.getvalue())

# An object we invented five seconds ago
class PrefixedLogger:
    def __init__(self, prefix: str) -> None:
        self.prefix = prefix

    def write(self, text: str) -> None:
        if text.strip():
            print(f"  {self.prefix} {text.strip()}")

write_report(PrefixedLogger("LOG:"), data)

# A collector that keeps the lines
class LineCollector:
    def __init__(self) -> None:
        self.lines: list[str] = []

    def write(self, text: str) -> None:
        self.lines.append(text.rstrip())

collector = LineCollector()
write_report(collector, data)
print("\ncollected:", collector.lines)

print("""
Three completely unrelated classes. No shared base, no registration,
no interface declaration. The only contract is: "has .write(str)".
""")

### ⚠️ `isinstance` checks fight duck typing

A common instinct when writing a "safe" function is to check the type first:

```python
def process(items):
    if not isinstance(items, list):
        raise TypeError("expected a list")
```

That looks defensive. It is actually **restrictive** — it rejects tuples, sets, generators
and every custom sequence, none of which would have caused a problem.

Two better options:

| Approach | When |
|---|---|
| **EAFP** — "Easier to Ask Forgiveness than Permission": just try it, catch the error | Almost always the Pythonic default |
| **LBYL** — "Look Before You Leap": check a *capability*, not a type | When the failure would be expensive or partial |

If you must check, check for the **capability** (`hasattr(x, "read")`) or the **abstract
type** (`isinstance(x, Iterable)`), not the concrete class.

In [ ]:
from collections.abc import Iterable, Sized


# ---- Too strict: rejects perfectly good inputs ----
def total_strict(items) -> int:
    if not isinstance(items, list):
        raise TypeError(f"expected a list, got {type(items).__name__}")
    return sum(items)


print("list  :", total_strict([1, 2, 3]))
for bad in [(1, 2, 3), {1, 2, 3}, range(4)]:
    try:
        total_strict(bad)
    except TypeError as exc:
        print(f"  rejected {type(bad).__name__:<9}: {exc}")


# ---- EAFP: just do it, and let the failure speak ----
def total_eafp(items) -> int:
    return sum(items)

print("\nEAFP accepts everything reasonable:")
for good in [[1, 2, 3], (1, 2, 3), {1, 2, 3}, range(4), (n for n in [1, 2, 3])]:
    print(f"  {type(good).__name__:<9} -> {total_eafp(good)}")


# ---- LBYL done right: check the CAPABILITY, not the class ----
def describe(value) -> str:
    if isinstance(value, Iterable) and not isinstance(value, (str, bytes)):
        n = len(value) if isinstance(value, Sized) else "unknown"
        return f"iterable with {n} items"
    return f"scalar {value!r}"

for v in [[1, 2], (1, 2, 3), {"a": 1}, "text", 42, (x for x in range(3))]:
    print(f"  {str(type(v).__name__):<9} -> {describe(v)}")


# ---- hasattr for a specific method ----
def close_quietly(resource) -> None:
    if hasattr(resource, "close"):
        resource.close()
        print(f"  closed {type(resource).__name__}")
    else:
        print(f"  {type(resource).__name__} has nothing to close")

import io
close_quietly(io.StringIO())
close_quietly([1, 2, 3])

---

## 2. When you *do* want a contract

Duck typing has one real weakness: **the contract is invisible**.

`write_report(destination, rows)` requires `destination` to have `.write(str)`. Where is
that written down? Only in the body of the function, and in the docstring if you remembered.
Nothing tells a caller in advance, no tool can check it, and the failure arrives at run time
deep inside the call.

Python offers two ways to make the contract explicit — and the difference between them is
the single most useful distinction in modern Python OOP.

| | **ABC** (`abc.ABC`) | **Protocol** (`typing.Protocol`) |
|---|---|---|
| Typing style | **Nominal** — you must *declare* the relationship | **Structural** — you must merely *fit* the shape |
| How you conform | Inherit from it | Have the right methods |
| Enforced when | At instantiation, at run time | By a type checker, before running |
| Works on classes you don't own | Only via `.register()` | ✅ Automatically |
| Analogy | A **club you join** | A **shape you happen to be** |

### Which one?

- **ABC** when you own the hierarchy and want to *force* subclasses to implement something,
  and to share some implementation with them.
- **Protocol** when you are describing what you *accept* — especially third-party or
  built-in types you cannot modify.

In [ ]:
from abc import ABC, abstractmethod


# ---- An ABC: a contract enforced at instantiation ----
class StorageBackend(ABC):
    """Every storage backend must implement save and load."""

    @abstractmethod
    def save(self, key: str, data: bytes) -> None: ...

    @abstractmethod
    def load(self, key: str) -> bytes: ...

    # ABCs may also provide shared implementation
    def save_text(self, key: str, text: str) -> None:
        self.save(key, text.encode("utf-8"))

    def load_text(self, key: str) -> str:
        return self.load(key).decode("utf-8")


class MemoryStorage(StorageBackend):
    def __init__(self) -> None:
        self._data: dict[str, bytes] = {}

    def save(self, key: str, data: bytes) -> None:
        self._data[key] = data

    def load(self, key: str) -> bytes:
        return self._data[key]


store = MemoryStorage()
store.save_text("greeting", "hello")          # inherited helper
print("round trip:", store.load_text("greeting"))


# ---- The enforcement: an incomplete subclass cannot be instantiated ----
class BrokenStorage(StorageBackend):
    def save(self, key: str, data: bytes) -> None:
        pass
    # load() is missing

try:
    BrokenStorage()
except TypeError as exc:
    print("\nincomplete subclass:", exc)

# ...and neither can the ABC itself
try:
    StorageBackend()
except TypeError as exc:
    print("abstract base      :", exc)


# ---- The limitation: a perfectly good class that did not inherit ----
class S3Storage:
    """Written by another team. Has exactly the right methods."""
    def __init__(self) -> None:
        self._data: dict[str, bytes] = {}

    def save(self, key: str, data: bytes) -> None:
        self._data[key] = data

    def load(self, key: str) -> bytes:
        return self._data[key]


print("\nS3Storage has the right shape, but:")
print("  isinstance(S3Storage(), StorageBackend) ->", isinstance(S3Storage(), StorageBackend))
print("  It works fine at run time - duck typing - but the ABC does not recognise it.")

# You can retrofit membership with register(), without inheriting
StorageBackend.register(S3Storage)
print("  after .register()                       ->", isinstance(S3Storage(), StorageBackend))
print("  (register() does NOT check the methods exist - it is a promise, not a proof)")

In [ ]:
from typing import Protocol, runtime_checkable


# ---- A Protocol: describes a SHAPE, nothing has to inherit it ----
class Storage(Protocol):
    """Anything with these methods satisfies Storage."""

    def save(self, key: str, data: bytes) -> None: ...
    def load(self, key: str) -> bytes: ...


class MemoryStore:
    def __init__(self) -> None:
        self._data: dict[str, bytes] = {}

    def save(self, key: str, data: bytes) -> None:
        self._data[key] = data

    def load(self, key: str) -> bytes:
        return self._data[key]


class LoggingStore:
    """Wraps another store - and is itself a Storage. Composition in action."""

    def __init__(self, inner: Storage) -> None:
        self._inner = inner

    def save(self, key: str, data: bytes) -> None:
        print(f"  SAVE {key} ({len(data)} bytes)")
        self._inner.save(key, data)

    def load(self, key: str) -> bytes:
        print(f"  LOAD {key}")
        return self._inner.load(key)


# The function declares WHAT IT NEEDS, not what it accepts
def cache_response(store: Storage, url: str, body: str) -> str:
    store.save(url, body.encode("utf-8"))
    return store.load(url).decode("utf-8")


plain = MemoryStore()
print("plain store  :", cache_response(plain, "/api/users", "[]"))

wrapped = LoggingStore(MemoryStore())
print("wrapped store:", cache_response(wrapped, "/api/users", "[]"))

print("\nNeither class inherits from Storage:")
print("  MemoryStore bases :", [b.__name__ for b in MemoryStore.__bases__])
print("  A type checker still accepts both - that is STRUCTURAL typing.")


# ---- @runtime_checkable lets isinstance() check the shape ----
@runtime_checkable
class Closeable(Protocol):
    def close(self) -> None: ...


class FileHandle:
    def close(self) -> None:
        print("  handle closed")


print("\nruntime_checkable:")
print("  isinstance(FileHandle(), Closeable) ->", isinstance(FileHandle(), Closeable))
print("  isinstance([1,2,3],     Closeable) ->", isinstance([1, 2, 3], Closeable))

print("""
  ⚠️ runtime_checkable only checks that the METHODS EXIST.
     It does not check signatures or return types - only a static
     checker does that.
""")

---

## 3. Putting it together: dependency injection

"Dependency injection" sounds like a framework term. In Python it is just this:

> **Pass a collaborator in, rather than constructing it inside.**

That one habit is what makes code testable, because the test can pass a fake. Combined with
composition (**5.2**) and Protocols, it is most of what "good design" means in practice —
and it needs no library at all.

In [ ]:
from typing import Protocol


class Clock(Protocol):
    def now(self) -> str: ...


class Storage(Protocol):
    def save(self, key: str, data: bytes) -> None: ...
    def load(self, key: str) -> bytes: ...


# ---- The thing under test: collaborators come IN ----
class SessionCache:
    def __init__(self, storage: Storage, clock: Clock) -> None:
        self._storage = storage         # injected
        self._clock = clock             # injected

    def put(self, session_id: str, user: str) -> None:
        record = f"{user}@{self._clock.now()}"
        self._storage.save(session_id, record.encode("utf-8"))

    def get(self, session_id: str) -> tuple[str, str]:
        user, _, when = self._storage.load(session_id).decode("utf-8").partition("@")
        return user, when


# ---- Production collaborators ----
class MemoryStore:
    def __init__(self) -> None:
        self._data: dict[str, bytes] = {}

    def save(self, key: str, data: bytes) -> None:
        self._data[key] = data

    def load(self, key: str) -> bytes:
        return self._data[key]


class SystemClock:
    def now(self) -> str:
        from datetime import datetime, timezone
        return datetime.now(timezone.utc).isoformat(timespec="seconds")


cache = SessionCache(MemoryStore(), SystemClock())
cache.put("abc123", "aditya")
print("production:", cache.get("abc123"))


# ---- Test collaborators: tiny, deterministic, no mocking library ----
class FrozenClock:
    def now(self) -> str:
        return "2024-01-01T00:00:00+00:00"


class FailingStore:
    def save(self, key: str, data: bytes) -> None:
        raise IOError("disk full")

    def load(self, key: str) -> bytes:
        raise KeyError(key)


test_cache = SessionCache(MemoryStore(), FrozenClock())
test_cache.put("abc123", "aditya")
print("\ndeterministic:", test_cache.get("abc123"))
assert test_cache.get("abc123") == ("aditya", "2024-01-01T00:00:00+00:00")
print("assertion passed - the timestamp is now testable")

# Failure paths become trivial to exercise
broken = SessionCache(FailingStore(), FrozenClock())
try:
    broken.put("abc123", "aditya")
except IOError as exc:
    print("\nfailure path:", exc)

print("""
Had SessionCache constructed its own MemoryStore() and called
datetime.now() directly, neither of those tests would be possible
without patching global state.
""")

---

## 4. Choosing: ABC, Protocol, or nothing

Most of the time the answer is **nothing** — duck typing with a clear docstring and type
hints is enough. Reach for the heavier tools when there is a concrete reason.

| Situation | Use |
|---|---|
| Small script, one implementation | **Nothing** — just call the method |
| Describing what a *parameter* must provide | **Protocol** |
| Third-party or builtin types must satisfy it | **Protocol** |
| You own a family of classes and want to *force* implementation | **ABC** |
| You want to share partial implementation with subclasses | **ABC** |
| You need `isinstance()` to work at run time | **ABC**, or `@runtime_checkable` Protocol |
| Both — enforce for your own subclasses, accept anything shaped right | ABC for the base, Protocol for the parameter |

### The design summary for folder 05

Four principles, in the order they should occur to you:

1. **Duck typing first.** Ask what an object can do, not what it is.
2. **Composition over inheritance.** "Has a" beats "is a" (**5.2**).
3. **Inject dependencies.** Pass collaborators in; don't construct them inside.
4. **Make contracts explicit — when it pays.** Protocol for what you accept, ABC for what
   you enforce.

> Everything here is about **coupling**. Inheritance couples classes tightly and permanently;
> composition and duck typing couple them loosely and temporarily. Loose coupling is what
> lets you change one part without the others noticing.

---

## Common Mistakes & Pitfalls

1. **Checking `isinstance(x, list)` when you only iterate `x`.** It rejects tuples, sets and generators for no reason. Accept `Iterable`, or just try it.
2. **Inheriting purely to satisfy an interface.** If you only need the methods, duck typing already works — a Protocol documents it without the coupling.
3. **Forgetting `@runtime_checkable`** and then calling `isinstance()` against a Protocol — `TypeError`.
4. **Expecting `@runtime_checkable` to verify signatures.** It only checks that the method *names* exist.
5. **Using `ABC.register()` and assuming it validated anything.** It does not — it is an unchecked promise that the class conforms.
6. **Constructing collaborators inside a class** (`self.db = PostgresClient()`). That class can now never be tested without a database.
7. **Adding an ABC for a single implementation.** An interface with one implementer is usually just indirection.
8. **Forgetting that `str` is `Iterable`.** `isinstance('abc', Iterable)` is `True`, so a string silently iterates character by character.

## Best Practices

- Default to **duck typing**; state the requirement in the docstring and the type hint.
- Prefer **EAFP** (`try`/`except`) over defensive type checks.
- Use **Protocol** to describe what a function *accepts*; **ABC** to enforce what your own subclasses must *provide*.
- Annotate parameters with the **most permissive** type that works — `Iterable` over `list`, a Protocol over a concrete class.
- **Inject dependencies** through `__init__` so tests can substitute fakes.
- Keep protocols narrow — one or two methods. A protocol with ten methods is a class in disguise.
- Write the fake implementations as real tiny classes; they are usually clearer than a mocking library.

## Practice Exercises

Try these before moving on.

1. Write `render(rows, out)` that works with a file, a `StringIO` and a custom collector.
2. Define a `Notifier` Protocol with `send(message)`, then write email, Slack and no-op implementations that share no base class.
3. Convert a class that constructs its own HTTP client into one that accepts it, then test it with a fake that returns canned responses.
4. Take an ABC with a single subclass and argue whether it earns its place.
5. Make a Protocol `@runtime_checkable` and show a class that passes `isinstance()` but would still fail — because the signature is wrong.
6. Use `ABC.register()` on a class missing a required method, and show that `isinstance` still returns `True`.
7. Rewrite the `StorageBackend` ABC as a Protocol. What did you gain, and what did you lose?
8. Write a decorator-free `LoggingStore` wrapper and explain why it is both a *user* of Storage and a Storage itself.